# 📐 Reducción de Dimensionalidad con PCA & Clasificación de Litofacies
## Módulo 4 · Día 3 · Compresión de Well Logs & Visualización 3D · Capacitación SLB

**Instructor: David Ponce**

---

### 🎯 ¿Qué aprenderemos hoy?

Los Días 1 y 2 aprendimos K-Means y DBSCAN. Pero nos quedó una duda:
**cuando tienes 5, 7 o más variables, ¿cómo visualizas los clusters?**

**PCA (Principal Component Analysis)** resuelve esto: toma tus 4, 7 o 100
variables y las **condensa** en 2 o 3 componentes principales que conservan
casi toda la información. Es como resumir un libro de 500 páginas en 2 frases.

Al final de esta clase, serás capaz de:
1. Explicar por qué NO es lo mismo seleccionar 2 variables que aplicar PCA
2. Reducir 4 curvas geofísicas (GR, ILD, RHOB, NPHI) a 2 componentes
3. Interpretar FÍSICAMENTE qué significa PC1 y PC2 en términos de geología
4. Aplicar K-Means sobre PCA y encontrar litofacies automáticamente
5. Crear visualizaciones 3D interactivas con Plotly

> 💡 **Tip:** Cada celda de código está comentada línea por línea. Si algo
> no te queda claro, ¡pregunta!

---
## 🧩 PARTE 1: Importando las herramientas

Hoy tenemos UNA librería nueva: `sklearn.decomposition.PCA`.
Y una herramienta de visualización NUEVA: `plotly.express` para gráficos 3D.

### 📦 Celda 1: Importación de librerías

In [ ]:
# ============================================
# CELDA 1: Importación de librerías
# ============================================

# ─── Herramientas clásicas (ya conocidas) ───
import numpy as np               # Cálculo numérico: np.sum(), np.cumsum()
import pandas as pd              # Tablas: pd.read_csv()
import matplotlib.pyplot as plt  # Gráficos base: plt.plot(), plt.bar()
import seaborn as sns            # Gráficos estadísticos bonitos

# ─── Herramienta NUEVA: Plotly para 3D ───
import plotly.express as px
#   ↑ 'plotly' = librería de visualización interactiva de última generación.
#   ↑ '.express' = módulo de alto nivel: gráficos en UNA línea de código.
#   ↑ 'as px' = alias estándar de Plotly Express.
#   ↑ Lo usaremos para px.scatter_3d(): gráfico 3D que PUEDES ROTAR con el mouse.

# ─── Herramientas de ML ───
from sklearn.preprocessing import StandardScaler
#   ↑ El escalador Z-Score. IGUAL de obligatorio que en K-Means y DBSCAN.

from sklearn.decomposition import PCA
#   ↑ '.decomposition' = módulo para descomposición y reducción de dimensionalidad.
#   ↑ 'PCA' = Principal Component Analysis. ¡El protagonista del día!
#   ↑ Toma N variables correlacionadas y las condensa en 2-3 componentes.

from sklearn.cluster import KMeans
#   ↑ Nuestro viejo conocido del Día 1. Lo aplicaremos SOBRE los resultados de PCA.

# ─── Estética ───
sns.set_theme(style="whitegrid")
print("✅ Todas las librerías importadas.")


### 🔍 ¿Qué hace cada librería NUEVA?

| Librería | ¿Qué es? | ¿Para qué la usamos HOY? |
|----------|----------|---------------------------|
| `plotly.express` | Gráficos interactivos profesionales | `px.scatter_3d()` para rotar un volumen 3D de litofacies |
| `PCA` | Algoritmo de reducción de dimensionalidad | Comprimir 4 curvas geofísicas → 2 componentes manteniendo ~85% de información |
| `KMeans` | Clustering (Día 1) | Aplicado SOBRE PC1 y PC2 para encontrar litofacies automáticamente |

> 🆕 **Plotly Express** es nuevo. A diferencia de matplotlib/seaborn que generan
> imágenes estáticas, Plotly crea gráficos **interactivos** que puedes rotar,
> hacer zoom, y explorar con el mouse.

---
## 📥 PARTE 2: Cargando los registros geofísicos

El dataset de hoy es `registros_pozo.csv`. Contiene **7,000 registros**
de un pozo, medidos cada 10 centímetros desde 1500m hasta 2200m de profundidad.

Cada registro tiene 4 curvas geofísicas:
- `GR_API`: Gamma Ray — mide radiactividad (distingue arcilla de arena)
- `ILD_ohm_m`: Resistividad — mide resistividad eléctrica (agua vs petróleo)
- `RHOB_g_cc`: Densidad Bulk — densidad de la roca (g/cc)
- `NPHI_v_v`: Porosidad Neutrón — estimador de porosidad

> 🧠 **El reto:** son solo 4 variables, pero YA no puedo verlas todas a la vez
> en un gráfico. Necesito PCA para condensarlas en 2 dimensiones.

### 📦 Celda 2: Descargar y cargar el dataset

In [ ]:
# ============================================
# CELDA 2: Cargar registros geofísicos
# ============================================

# ─── Descargar desde GitHub ───
!wget -q https://raw.githubusercontent.com/DavidPonce84/machine-learning-course/main/modulo_4_no_supervisado/data/registros_pozo.csv -O registros_pozo.csv
#   ↑ '!' = comando de Linux en Colab.
#   ↑ 'wget -q' = descarga silenciosa.

# ─── Cargar en DataFrame ───
df_logs = pd.read_csv('registros_pozo.csv')
#   ↑ 'df_logs' = DataFrame con los registros. "logs" = well logs.

# ─── Ver primeras filas ───
df_logs.head()
#   ↑ Verifica: Profundidad_m, GR_API, ILD_ohm_m, RHOB_g_cc, NPHI_v_v.


### 📦 Celda 3: Exploración inicial

In [ ]:
# ============================================
# CELDA 3: Exploración del dataset
# ============================================

print("📊 Dimensiones (filas, columnas):", df_logs.shape)
#   ↑ Esperamos (7001, 5): ~7,000 registros x 5 columnas.

print("\n📋 Tipos de datos:")
print(df_logs.dtypes)
#   ↑ Todas DEBEN ser float64 (números decimales).

print("\n📈 Estadísticas:")
df_logs.describe()
#   ↑ Mira los rangos: GR va de ~10 a ~120 API, RHOB de ~1.0 a ~2.7 g/cc.
#   ↑ Si RHOB tiene min=1.0 → probablemente hay washouts que limpiar.
#   ↑ Si ILD tiene max > 100 → hay zonas de alta resistividad (¿petróleo?).


---
## 📏 PARTE 3: Escalando las 4 curvas

**Misma regla de siempre:** sin escalar, las variables con números más grandes
dominan. GR está en decenas (10-120), ILD en centenas (1-200+), RHOB en
unidades (1.0-2.7). Si no escalamos, ILD aplasta a las demás.

Además, **PCA requiere escalado** porque busca las direcciones de máxima
varianza. Si una variable tiene varianza 10,000 y otra 0.01, PCA solo 'verá'
la primera.

### 📦 Celda 4: Escalar con StandardScaler

In [ ]:
# ============================================
# CELDA 4: Escalar las 4 curvas geofísicas
# ============================================

# ─── Definir las columnas a usar ───
log_features = ['GR_API', 'ILD_ohm_m', 'RHOB_g_cc', 'NPHI_v_v']
#   ↑ Lista con las 4 curvas geofísicas.
#   ↑ 'log_features' = "features de registros (logs)".

# ─── Escalar ───
scaler = StandardScaler()
#   ↑ Crea el escalador (aún no hace nada).

X_logs_scaled = scaler.fit_transform(df_logs[log_features])
#   ↑ 'df_logs[log_features]' = selecciona SOLO las 4 columnas.
#   ↑ '.fit_transform()' = calcula μ, σ y aplica z = (x-μ)/σ.
#   ↑ 'X_logs_scaled' = array de 7000×4 con media=0, std=1.

# ─── Verificar ───
print("✅ Datos escalados. Forma:", X_logs_scaled.shape)
print("   Media GR (debe ser ≈0):", X_logs_scaled[:, 0].mean().round(6))
print("   Std  GR (debe ser ≈1):", X_logs_scaled[:, 0].std().round(6))
#   ↑ '[:, 0]' = todas las filas, columna 0 (GR_API).


---
## 🔬 PARTE 4: Aplicando PCA — De 4 dimensiones a 2

Aquí está el corazón del día. PCA va a CREAR 2 variables NUEVAS (PC1, PC2)
que son **combinaciones** de las 4 originales. No selecciona 2 — CREA 2.

### 🧠 ¿Qué va a pasar?
1. PCA analiza las 4 curvas y encuentra la dirección de MÁXIMA varianza → PC1
2. Luego busca la SEGUNDA dirección (perpendicular a PC1) → PC2
3. Proyecta cada registro sobre PC1 y PC2 → cada punto ahora tiene 2 coordenadas

> 🔦 **Analogía de la sombra:** Tus datos 4D son como un objeto 3D. PCA elige
> el MEJOR ÁNGULO para proyectar una sombra 2D que conserve la máxima información.

### 📦 Celda 5: Entrenar PCA y ver la varianza explicada

In [ ]:
# ============================================
# CELDA 5: Aplicar PCA — Reducir 4D → 2D
# ============================================

# ─── Paso 1: Crear y entrenar PCA ───
pca = PCA(n_components=2)
#   ↑ 'PCA(n_components=2)' = "quiero reducir a 2 componentes".
#   ↑ Puedes poner 3 para 3D, o 0.95 para "conserva el 95% de varianza".

X_pca = pca.fit_transform(X_logs_scaled)
#   ↑ '.fit()' = analiza las 4 curvas y encuentra PC1 y PC2.
#   ↑ '.transform()' = proyecta los datos originales sobre PC1 y PC2.
#   ↑ 'X_pca' = array de 7000×2. Columna 0 = PC1, columna 1 = PC2.

# ─── Paso 2: Ver cuánta información conservamos ───
print("📊 Varianza explicada por componente:")
print(f"   PC1: {pca.explained_variance_ratio_[0]:.1%}")
#   ↑ 'explained_variance_ratio_' = array con el % de varianza de cada componente.
#   ↑ '[0]' = primer componente (PC1).
print(f"   PC2: {pca.explained_variance_ratio_[1]:.1%}")
#   ↑ '[1]' = segundo componente (PC2).
print(f"   TOTAL conservado: {pca.explained_variance_ratio_.sum():.1%}")
#   ↑ '.sum()' = suma los porcentajes. PC1 + PC2 = % total de información conservada.
#   ↑ Ejemplo típico: PC1=55%, PC2=30%, TOTAL=85%.

# ─── Paso 3: Agregar PC1 y PC2 al DataFrame ───
df_logs['PC1'] = X_pca[:, 0]
#   ↑ '[:, 0]' = todas las filas, columna 0 = PC1.
#   ↑ Crea una columna NUEVA en el DataFrame.
df_logs['PC2'] = X_pca[:, 1]
#   ↑ '[:, 1]' = todas las filas, columna 1 = PC2.

df_logs[['Profundidad_m', 'PC1', 'PC2']].head()
#   ↑ Muestra profundidad + las 2 nuevas columnas.


### 📦 Celda 6: Scree Plot — Visualizando la varianza explicada

In [ ]:
# ============================================
# CELDA 6: Scree Plot — ¿Cuánta información aporta cada PC?
# ============================================

# ─── Entrenar PCA con TODOS los componentes posibles ───
pca_full = PCA().fit(X_logs_scaled)
#   ↑ Sin 'n_components' → usa tantos componentes como variables originales (4).

# ─── Graficar barras de varianza explicada ───
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
#   ↑ 'subplots(1, 2)' = 1 fila, 2 columnas de gráficos lado a lado.

# --- Gráfico 1: Varianza individual ---
components = [f'PC{i+1}' for i in range(len(log_features))]
#   ↑ Crea la lista ['PC1', 'PC2', 'PC3', 'PC4'].

ax[0].bar(components, pca_full.explained_variance_ratio_, color=['#38bdf8','#f59e0b','#94a3b8','#94a3b8'])
#   ↑ Gráfico de BARRAS. Altura = % de varianza de cada componente.
#   ↑ PC1 y PC2 en colores destacados, PC3 y PC4 grises (poco aportan).
ax[0].set_title('Varianza Explicada por Componente')
ax[0].set_ylabel('% de Varianza')
for i, v in enumerate(pca_full.explained_variance_ratio_):
    ax[0].text(i, v + 0.01, f'{v:.1%}', ha='center', fontweight='bold')
#   ↑ Agrega el porcentaje SOBRE cada barra.

# --- Gráfico 2: Varianza acumulada ---
cumsum = np.cumsum(pca_full.explained_variance_ratio_)
#   ↑ 'np.cumsum()' = suma acumulativa: [PC1, PC1+PC2, PC1+PC2+PC3, ...].
ax[1].plot(range(1, 5), cumsum, 'o-', color='#22c55e', linewidth=2, markersize=10)
ax[1].axhline(y=0.85, color='red', linestyle='--', label='85% (umbral típico)')
ax[1].set_title('Varianza Acumulada')
ax[1].set_xlabel('Número de Componentes')
ax[1].set_ylabel('% Acumulado')
ax[1].legend()
ax[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

print(f"✅ Con 2 componentes conservamos el {cumsum[1]:.1%} de la información.")
print(f"   El {pca_full.explained_variance_ratio_[2]:.1%} restante en PC3 es mayormente ruido.")


### 🔍 ¿Cómo leer el Scree Plot?

- **Gráfico izquierdo (barras):** PC1 es la barra más alta — captura la mayor
  variación. PC2 es la segunda. PC3 y PC4 son pequeñas — añaden poco.
- **Gráfico derecho (línea):** La línea sube rápido con PC1+PC2 y luego se aplana.
  El punto donde cruza 85% te dice cuántos componentes necesitas.

> 🧠 **Regla práctica:** Quédense con los componentes que sumen >80-90% de
> varianza acumulada. El resto es ruido o redundancia.

---
## 🧠 PARTE 5: Interpretando FÍSICAMENTE PC1 y PC2

PCA nos dio 2 números por registro. Pero... ¿qué SIGNIFICAN en términos
de geología? Vamos a descubrirlo viendo cómo contribuye cada curva original.

### 📦 Celda 7: Visualizar PC1 vs PC2 + Interpretación geológica

In [ ]:
# ============================================
# CELDA 7: Graficar PC1 vs PC2 e interpretar
# ============================================

# ─── Scatterplot de los 2 componentes principales ───
plt.figure(figsize=(10, 7))
plt.scatter(df_logs['PC1'], df_logs['PC2'], c=df_logs['GR_API'], cmap='viridis', alpha=0.5, s=1)
#   ↑ 'c=df_logs['GR_API']' = colorea cada punto según su valor de Gamma Ray.
#   ↑ 'cmap='viridis'' = paleta de colores (amarillo=alto GR, morado=bajo GR).
#   ↑ 'alpha=0.5' = transparencia. 's=1' = tamaño de punto pequeño (son 7,000).

plt.colorbar(label='Gamma Ray (API)')
#   ↑ Barra de colores: muestra la escala de GR.

plt.xlabel('PC1 — Componente Principal 1')
plt.ylabel('PC2 — Componente Principal 2')
plt.title('Well Logs proyectados sobre PC1 y PC2\nColoreados por Gamma Ray (GR)')
plt.grid(True, alpha=0.3)
plt.show()

# ─── ¿Cómo contribuye cada curva original a los componentes? ───
print("📊 Contribución de cada curva original a los componentes:")
loadings = pd.DataFrame(
    pca.components_.T,
    #   ↑ '.components_' = matriz de "cargas" (loadings).
    #   ↑ '.T' = transpuesta: filas=variables, columnas=componentes.
    columns=['PC1', 'PC2'],
    index=log_features
)
print(loadings.round(3))
#   ↑ Valores positivos = correlación directa con el componente.
#   ↑ Valores negativos = correlación inversa.
#   ↑ Magnitud = importancia de esa variable en ese componente.


### 🔍 Interpretación física esperada

Típicamente en well logs:
- **PC1:** GR contribuye FUERTE (positivo o negativo). Separa ARCILLAS (GR alto)
  de ARENAS (GR bajo). PC1 es el 'eje litológico'.
- **PC2:** ILD (resistividad) contribuye fuerte. Separa ARENAS CON AGUA
  (ILD bajo) de ARENAS CON PETRÓLEO (ILD alto). PC2 es el 'eje de fluidos'.

> 🛢️ **Esto es enorme:** PCA 'aprendió' automáticamente los conceptos de
> litología y fluidos SIN que nadie le enseñara geología. Solo mirando
> cómo varían juntas las 4 curvas.

---
## 🎯 PARTE 6: K-Means sobre PCA — Encontrando Litofacies

Ahora que tenemos los datos en 2D (PC1, PC2), aplicamos K-Means.
¿La ventaja? Los clusters que encontremos tendrán SIGNIFICADO GEOLÓGICO.

### 📦 Celda 8: K-Means sobre los componentes principales

In [ ]:
# ============================================
# CELDA 8: K-Means sobre PCA para clasificar litofacies
# ============================================

# ─── Aplicar K-Means con K=3 (3 tipos de roca esperados) ───
kmeans_facies = KMeans(n_clusters=3, random_state=42)
#   ↑ 'n_clusters=3' = buscamos 3 litofacies: arena con petróleo, arena con agua, arcilla.
#   ↑ 'random_state=42' = semilla fija para reproducibilidad.

df_logs['Facies'] = kmeans_facies.fit_predict(X_pca)
#   ↑ 'fit_predict(X_pca)' = entrena K-Means sobre PC1 y PC2 y asigna cluster.
#   ↑ 'df_logs['Facies']' = NUEVA columna con la litofacie asignada (0, 1, o 2).

# ─── Ver el perfil promedio de cada facies ───
print("📊 Perfil promedio de cada litofacie:")
df_logs.groupby('Facies')[log_features].mean().round(2)
#   ↑ Agrupa por facies y calcula la MEDIA de cada curva geofísica.
#   ↑ Esto te dice: "la Facies 0 tiene GR≈30, ILD≈35 → arena con petróleo".


### 🔍 ¿Qué esperamos ver?

| Facies | GR | ILD | RHOB | NPHI | Interpretación |
|--------|-----|------|------|------|----------------|
| 0 | Bajo (~30) | Alto (~35) | Medio (~2.35) | Medio (~0.20) | Arena con PETRÓLEO |
| 1 | Bajo (~35) | Bajo (~5) | Medio (~2.40) | Medio (~0.18) | Arena con AGUA |
| 2 | Alto (~90) | Bajo (~2) | Alto (~2.55) | Bajo (~0.10) | ARCILLA (sello) |

> 🎯 **K-Means + PCA encontraron automáticamente las 3 litofacies** que un
> petrofísico identificaría manualmente. Pero en SEGUNDOS y para 7,000 registros.

---
## 🌐 PARTE 7: Visualización 3D Interactiva con Plotly

Matplotlib nos da imágenes estáticas. Plotly nos da un gráfico 3D que
**podemos rotar, hacer zoom y explorar** con el mouse. Esto es invaluable
para inspeccionar las litofacies a lo largo de la profundidad del pozo.

### 📦 Celda 9: Gráfico 3D interactivo con Plotly Express

In [ ]:
# ============================================
# CELDA 9: Visualización 3D interactiva con Plotly
# ============================================

# ─── Crear scatterplot 3D ───
fig = px.scatter_3d(
    df_logs,                          # ← DataFrame fuente
    x='PC1',                          # ← Eje X: Componente Principal 1
    y='PC2',                          # ← Eje Y: Componente Principal 2
    z='Profundidad_m',                # ← Eje Z: ¡Profundidad real del pozo!
    color='Facies',                   # ← Colorea por litofacie (0, 1, 2)
    opacity=0.6,                      # ← Transparencia para ver puntos solapados
    title='Clasificación 3D de Litofacies — PCA + K-Means',
    labels={'PC1': 'Componente 1 (Litología)',
            'PC2': 'Componente 2 (Fluidos)',
            'Profundidad_m': 'Profundidad (m)',
            'Facies': 'Litofacie'}
    #   ↑ 'labels' = renombra los ejes para que sean más descriptivos.
)

# ─── Configurar el eje Z invertido (profundidad) ───
fig.update_scenes(zaxis_autorange="reversed")
#   ↑ La profundidad aumenta hacia ABAJO. Invertimos el eje Z para que el
#   ↑ gráfico se vea como un pozo real: superficie arriba, fondo abajo.

# ─── Ajustar tamaño de puntos ───
fig.update_traces(marker_size=2)
#   ↑ Puntos pequeños porque son 7,000 registros.

fig.show()
#   ↑ ¡Aquí aparece la magia! Un gráfico 3D que puedes ROTAR con el mouse.
#   ↑ Prueba: arrastra para rotar, scroll para zoom, doble click para reset.


### 🔍 ¿Qué deberías observar?

- **3 'nubes' de colores distintos** a lo largo de la profundidad
- Cada color = una litofacie (arena con petróleo, arena con agua, arcilla)
- Las arcillas (Facies 2) suelen estar en zonas específicas (sellos)
- Las arenas con petróleo (Facies 0) aparecen en zonas de interés

> 🖱️ **Interactúa:** Rota el gráfico para ver el pozo desde diferentes ángulos.
> Haz zoom en las zonas de interés. Pasa el mouse sobre los puntos para ver
> sus coordenadas exactas.

---
## 🎨 PARTE 8: Visualización 2D de las litofacies encontradas

También podemos graficar en 2D (PC1 vs PC2) coloreado por la litofacie
que K-Means asignó.

### 📦 Celda 10: Plano PC1-PC2 coloreado por litofacie

In [ ]:
# ============================================
# CELDA 10: Visualizar litofacies en el plano PCA
# ============================================

plt.figure(figsize=(10, 7))

# ─── Scatterplot coloreado por Facies ───
scatter = plt.scatter(
    df_logs['PC1'], df_logs['PC2'],
    c=df_logs['Facies'],             # ← Color según litofacie (0, 1, 2)
    cmap='viridis',                  # ← Mapa de colores
    alpha=0.5, s=1                   # ← Transparencia, tamaño
)
plt.colorbar(scatter, label='Litofacie')
#   ↑ Barra de colores mostrando la correspondencia color → facies.

plt.xlabel('PC1 — Componente Litológico (Arcilla ↔ Arena)')
plt.ylabel('PC2 — Componente de Fluidos (Agua ↔ Petróleo)')
plt.title('Litofacies identificadas por PCA + K-Means')
plt.grid(True, alpha=0.3)
plt.show()

# ─── Resumen final ───
print("📊 Conteo de registros por litofacie:")
print(df_logs['Facies'].value_counts())
print(f"\n✅ PCA redujo 4 dimensiones → 2, conservando {pca.explained_variance_ratio_.sum():.1%} de información.")
print(f"✅ K-Means (K=3) identificó 3 litofacies sobre los componentes principales.")


---
## 📋 RECAP: El Pipeline PCA Completo

```
┌──────────────────────────────────────────────────────────┐
│ 1. IMPORTAR herramientas  → PCA, KMeans, Plotly          │
│ 2. CARGAR well logs       → pd.read_csv()                │
│ 3. EXPLORAR               → .shape, .describe()          │
│ 4. ESCALAR                → StandardScaler()             │
│ 5. APLICAR PCA            → PCA(n_components=2)          │
│ 6. EVALUAR varianza       → explained_variance_ratio_    │
│ 7. INTERPRETAR PC1, PC2   → loadings, significado físico │
│ 8. K-MEANS sobre PCA      → KMeans(n_clusters=3)         │
│ 9. VISUALIZAR 3D          → px.scatter_3d()              │
│ 10. VISUALIZAR 2D         → plt.scatter()                │
└──────────────────────────────────────────────────────────┘
```

### ✅ Lo que aprendiste hoy

- PCA NO selecciona variables — CREA nuevas que son combinaciones de las originales
- PC1 y PC2 típicamente conservan >80% de la información
- Los componentes tienen SIGNIFICADO FÍSICO: litología, fluidos
- K-Means + PCA encuentra litofacies automáticamente
- Plotly permite visualizar el pozo completo en 3D interactivo

> 🚀 **Fin del Módulo 4.** Has aprendido K-Means, DBSCAN, y PCA. Tres
> herramientas fundamentales de Machine Learning No Supervisado aplicadas
> a problemas reales de la industria petrolera.
